# RepairScope-X — Notebook 24
## Reproducible cross-model repair atlas and evidence audit

**Purpose.** Reconstruct the reported repair results from their original, frozen raw trial records and evaluate which findings survive problem-level statistical comparisons and conservative verification checks.

**Data embedded in this notebook:**
- Dream: Notebook 12 exploratory (240 trials) and Notebook 13 independent (150 trials; primary and secondary preserved separately).
- ALoDLM 1.7B FP32: Notebook 15 entropy repair (64) and Notebook 16 left1 decoder ablation (64). The entropy copy inside Notebook 16 is **excluded** to avoid double counting.
- MDLM 0.5B: Notebook 18 repair (64) and Notebook 21 controlled-denoising **generations only** (48); Notebook 21 repairs are pending.
- BD3LM 0.5B: Notebook 19 repair (64) and Notebook 23 constant-per-block repair (64).
- Source generation corpora and the existing 140-case manual-review queue.

**Total already completed repair trials:** **710**, before separately considering any optional future MDLM Notebook 21 repair results.

### Interpretation rules
1. Do not pool results across architectures, model sizes, decoder policies, precisions or distinct benchmark cohorts into a single success percentage.
2. Count distinct **problem IDs** as the independent unit for confidence intervals and comparisons; repeated repair seeds are not additional problems.
3. Separate reported task success, numeric-answer match, explicit final-answer format and presence of a mathematically valid Countdown expression. A correct expression *mentioned* in an answer is **not necessarily a valid final response**.
4. Treat in-sample per-instance oracle repair as an **upper bound**, not a deployable selector.
5. The original MDLM 21 controlled-denoising repair study is **not included** until its results ZIP is provided.

The notebook uses Python's standard library and optional Matplotlib for figures. No Hugging Face models, tokens, installation, GPU, or user uploads are required.

Frozen evidence-bundle SHA-256: `ce4301a1201354e57b7c247fec4f24870ccc5d2edc854fa726e51d639861905e`.

In [ ]:
from pathlib import Path
import sys,os,json,csv,io,zipfile,hashlib,base64,math,re,ast,collections,statistics,random
from collections import Counter,defaultdict
from datetime import datetime,timezone

# Use Colab's /content directory when available; otherwise use local working directory.
ROOT=Path('/content') if Path('/content').exists() else Path.cwd()
OUT=ROOT/'RepairScopeX_NB24_Audit_Outputs'
OUT.mkdir(parents=True,exist_ok=True)
print('Python:',sys.version.split()[0]);print('Output:',OUT)

In [ ]:
BUNDLE_SHA256='ce4301a1201354e57b7c247fec4f24870ccc5d2edc854fa726e51d639861905e'
BUNDLE_B64='''...'''
blob=base64.b64decode(BUNDLE_B64)
assert hashlib.sha256(blob).hexdigest()==BUNDLE_SHA256
with zipfile.ZipFile(io.BytesIO(blob)) as z:
    names=z.namelist()
    assert len(names)==16 and 'provenance.json' in names
    INPUTS={name:z.read(name) for name in names}
PROVENANCE=json.loads(INPUTS['provenance.json'])
for key,item in PROVENANCE.items():
    suffix='csv' if item['kind']=='csv' else 'jsonl'
    assert hashlib.sha256(INPUTS[f'{key}.{suffix}']).hexdigest()==item['sha256'],key
print('15 original evidence files verified; bundle SHA:',BUNDLE_SHA256)

In [ ]:
def load_jsonl(key):
    return [json.loads(s) for s in INPUTS[f'{key}.jsonl'].decode('utf-8').splitlines() if s.strip()]

def write_csv(name, rows):
    p=OUT/name
    if not rows:
        p.write_text('')
        return p
    keys=list(dict.fromkeys(k for r in rows for k in r.keys()))
    with p.open('w',encoding='utf-8',newline='') as f:
        w=csv.DictWriter(f,fieldnames=keys,extrasaction='ignore');w.writeheader()
        for r in rows:w.writerow({k:(json.dumps(v,ensure_ascii=False) if isinstance(v,(list,dict)) else v) for k,v in r.items()})
    return p

def write_json(name,value):
    p=OUT/name;p.write_text(json.dumps(value,indent=2,ensure_ascii=False,allow_nan=False),encoding='utf-8');return p

REPAIR_KEYS=['dream12','dream13','alodlm15','alodlm16_left1','mdlm18','bd3lm19','bd3lm23']
raw_cohorts={k:load_jsonl(k) for k in REPAIR_KEYS}
assert {k:len(v) for k,v in raw_cohorts.items()}=={
 'dream12':240,'dream13':150,'alodlm15':64,'alodlm16_left1':64,
 'mdlm18':64,'bd3lm19':64,'bd3lm23':64
}
assert sum(map(len,raw_cohorts.values()))==710
assert all(x.get('status')=='ok' for v in raw_cohorts.values() for x in v)
for cohort,records in raw_cohorts.items():
    ids=[x.get('trial_id',x.get('id')) for x in records]
    assert all(ids) and len(ids)==len(set(ids)),f'duplicate trial IDs in {cohort}'
print('All 710 repair trials loaded and validated, without duplicate cohort IDs.')

In [ ]:
# Freeze original problem statements, target numbers, and generation outcomes.
NATURAL_KEYS=['dream11natural','alodlm14natural','mdlm18natural',
 'bd3lm19natural','mdlm21natural','bd3lm23natural','mdlm20length']
natural={k:load_jsonl(k) for k in NATURAL_KEYS}
problem_index=defaultdict(list)
for source,records in natural.items():
    for r in records:
        if r.get('problem_id') and r.get('prompt'):
            problem_index[r['problem_id']].append({'prompt':r['prompt'],'target':r.get('target'),'source':source})
print('Natural generation source counts:',{k:len(v) for k,v in natural.items()})
print('Problem IDs with reference prompts:',len(problem_index))

# The 140-case review file is a review QUEUE, not 140 adjudicated labels.
review_queue=list(csv.DictReader(io.StringIO(INPUTS['manual140.csv'].decode('utf-8'))))
assert len(review_queue)==140
print('Original unadjudicated cases:',len(review_queue))

## 1. A harmonized trial ledger

The ledger retains the original source cohort, decoding policy, native repair mechanism, model scale, precision, benchmark role, reported metrics and compute units. Do not equate Dream denoising NFEs with ALoDLM loop-token passes.

In [ ]:
COHORTS={
 'dream12':('Dream 7B','Dream exploratory','4bit NF4','mask-reopen','native diffusion'),
 'dream13':('Dream 7B','Dream independent','4bit NF4','mask-reopen','native diffusion'),
 'alodlm15':('ALoDLM 1.7B','ALoDLM entropy','fp32','suffix-regenerate','entropy'),
 'alodlm16_left1':('ALoDLM 1.7B','ALoDLM left1','fp32','suffix-regenerate','left1'),
 'mdlm18':('MDLM 0.5B','MDLM original 64','checkpoint fp16','native infill','low_confidence'),
 'bd3lm19':('BD3LM 0.5B','BD3LM original 64','checkpoint fp16','suffix-regenerate','low_confidence'),
 'bd3lm23':('BD3LM 0.5B','BD3LM constant per block','checkpoint fp16','suffix-regenerate','low_confidence')
}

ledger=[]
for key,rr in raw_cohorts.items():
    model,label,precision,method,decoder=COHORTS[key]
    for r in rr:
        role=r.get('confirmatory_role') if key=='dream13' else ('exploratory' if key=='dream12' else 'model-native')
        tid=r.get('trial_id') or r.get('id')
        ver=r.get('verification') or {}
        target=ver.get('target')
        if target is None and r.get('problem_id') in problem_index:
            target=problem_index[r['problem_id']][0].get('target')
        original=r.get('original_output','')
        repaired=r.get('repaired_output',r.get('output',''))
        compute=r.get('actual_forward_calls',r.get('nfe'))
        recurrence=r.get('same_wrong_numeric',r.get('wrong_answer_recurrence_numeric'))
        token_pres=r.get('token_position_preservation',r.get('preservation'))
        ledger.append({
            'cohort_key':key,'cohort':label,'model':model,'precision':precision,
            'native_intervention':method,'decoder_policy':decoder,
            'role':role,'trial_id':tid,'problem_id':r.get('problem_id'),
            'dataset':r.get('dataset'),'action':r.get('action'),
            'task_success_reported':int(bool(r.get('task_success'))),
            'numeric_success_reported':int(bool(r.get('numeric_success'))),
            'strict_success_reported':int(bool(r.get('strict_success'))),
            'same_wrong_numeric_reported':int(bool(recurrence)) if recurrence is not None else None,
            'token_preservation':float(token_pres) if token_pres is not None else None,
            'forward_calls_native':int(compute) if compute is not None else None,
            'loop_token_passes':r.get('loop_token_passes'),
            'wall_seconds':r.get('wall_seconds'),
            'target':target,
            'original_output':original,
            'repaired_output':repaired,
            'status':r.get('status')
        })
assert len(ledger)==710
assert all(r['problem_id'] for r in ledger)
write_csv('table01_all_710_repair_trials.csv',ledger)
print('Harmonized ledger:',len(ledger),'rows')

## 2. Task-verification sensitivity audit

This notebook **does not silently overwrite the original task labels**. Instead it independently checks whether a Countdown response contains at least one arithmetic expression that both (a) uses exactly the source numbers and (b) actually equals the target.

A valid expression appearing anywhere is only a **witness**, not a certified final response. GSM8K and SVAMP can be checked for explicit `FINAL_ANSWER` formatting, but their reasoning correctness requires further evidence or human review.

In [ ]:
FINAL_NUMERIC=re.compile(r'(?im)^[ \t]*FINAL_ANSWER\s*:\s*([-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?)\s*$')
NUMERIC=re.compile(r'[-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?')
ARITH_CANDIDATE=re.compile(r'(?<![\w.])(?:\(?\s*-?\d+(?:\.\d+)?\s*\)?\s*[-+*/]\s*)+\(?\s*-?\d+(?:\.\d+)?\s*\)?(?![\w.])')
PROMPT_NUMBERS=re.compile(r'Use the numbers\s+(.*?)\s+exactly once',re.I|re.S)

def eq(a,b):
    try:return math.isfinite(float(a)) and math.isfinite(float(b)) and math.isclose(float(a),float(b),abs_tol=1e-8,rel_tol=1e-8)
    except (TypeError,ValueError):return False

def validated_expr(expression):
    try:tree=ast.parse(expression,mode='eval')
    except (SyntaxError,ValueError):return None
    nums=[]
    def visit(n):
        if isinstance(n,ast.Expression):return visit(n.body)
        if isinstance(n,ast.Constant) and type(n.value) in (float,int):
            nums.append(float(n.value));return float(n.value)
        if isinstance(n,ast.UnaryOp) and isinstance(n.op,(ast.UAdd,ast.USub)):
            v=visit(n.operand);return v if isinstance(n.op,ast.UAdd) else -v
        if isinstance(n,ast.BinOp) and isinstance(n.op,(ast.Add,ast.Sub,ast.Mult,ast.Div)):
            a,b=visit(n.left),visit(n.right)
            return a+b if isinstance(n.op,ast.Add) else a-b if isinstance(n.op,ast.Sub) else a*b if isinstance(n.op,ast.Mult) else a/b
        raise ValueError('non-arithmetic operation')
    try:
        val=visit(tree)
        return val,nums if math.isfinite(val) else None
    except (ValueError,ZeroDivisionError,OverflowError):return None

def find_source_nums(row):
    for context in problem_index.get(row['problem_id'],[]):
        m=PROMPT_NUMBERS.search(context['prompt'])
        if m:
            nums=[float(x) for x in NUMERIC.findall(m.group(1))]
            if nums:return nums
    # For Dream, original verification exposes source_numbers.
    return None

def expr_witness(text,source_nums,target):
    if not source_nums or target is None:return None
    text=text.replace('×','*').replace('÷','/').replace('\\times','*')
    for m in ARITH_CANDIDATE.finditer(text):
        expression=m.group(0).strip()
        parsed=validated_expr(expression)
        if parsed is None:continue
        value,nums=parsed
        if len(nums)==len(source_nums) and all(eq(a,b) for a,b in zip(sorted(nums),sorted(source_nums))) and eq(value,target):
            return expression
    return None

assert validated_expr('2 + 24 + 28')[0]==54
assert validated_expr('29 + 9 + 8')[0]==46
assert validated_expr('__import__("os")') is None
assert expr_witness('2 + 24 + 28 = 54',[2,24,28],54)=='2 + 24 + 28'
assert expr_witness('29 + 9 + 8 = 12',[29,9,8],12) is None
print('Arithmetic safety and correctness tests: PASS')

In [ ]:
sensitivity=[]
for r in ledger:
    repaired=r['repaired_output'] or ''
    labels=FINAL_NUMERIC.findall(repaired)
    final= float(labels[-1]) if labels else None
    srcnums=find_source_nums(r) if r['dataset']=='countdown_fresh' else None
    witness=expr_witness(repaired,srcnums,r['target']) if r['dataset']=='countdown_fresh' else None
    sensitivity.append({
        'trial_id':r['trial_id'],'cohort':r['cohort'],'dataset':r['dataset'],
        'problem_id':r['problem_id'],'reported_task_success':r['task_success_reported'],
        'reported_numeric_success':r['numeric_success_reported'],
        'explicit_final_numeric':final,
        'explicit_final_correct':int(eq(final,r['target'])) if final is not None and r['target'] is not None else 0,
        'exact_expression_witness':witness or '',
        'exact_expression_witness_found':int(bool(witness)),
        'source_numbers_resolved':int(srcnums is not None) if r['dataset']=='countdown_fresh' else '',
        'requires_human_adjudication':int(r['dataset']=='countdown_fresh' and (bool(witness) != bool(r['task_success_reported'])))
    })
write_csv('table02_verifier_sensitivity.csv',sensitivity)
print('Mismatches requiring review:',sum(x['requires_human_adjudication'] for x in sensitivity))
print('Important: witness presence is NOT a final-answer grade.')

## 3. Cohort-specific action surfaces and confidence intervals

The Wilson 95% intervals below use **independent failed problems**, with one binary indicator per problem: did *any* seed succeed for that action? We also retain the trial success rate, but do not attach a trial-level inferential claim to repeated seeds.

In [ ]:
def wilson(k,n,z=1.959963984540054):
    if n<=0:return (None,None)
    p=k/n;den=1+z*z/n
    center=(p+z*z/(2*n))/den
    radius=(z/den)*math.sqrt(p*(1-p)/n+z*z/(4*n*n))
    return max(0,center-radius),min(1,center+radius)

def problem_action_rows(rows):
    by=defaultdict(list)
    for r in rows:by[(r['problem_id'],r['action'])].append(r)
    return {k:{'trial_success':sum(x['task_success_reported'] for x in rr)/len(rr),
                'any_success':int(any(x['task_success_reported'] for x in rr)),'seeds':len(rr)} for k,rr in by.items()}

surfaces=[]
for key in REPAIR_KEYS:
    subset=[r for r in ledger if r['cohort_key']==key and (key!='dream13' or r['role']=='primary_unseen_problem')]
    for action in sorted({r['action'] for r in subset}):
        trials=[r for r in subset if r['action']==action]
        problems=defaultdict(list)
        for r in trials:problems[r['problem_id']].append(r['task_success_reported'])
        succ=sum(any(v) for v in problems.values());n=len(problems)
        low,hi=wilson(succ,n)
        pres=[r['token_preservation'] for r in trials if r['token_preservation'] is not None]
        comp=[r['forward_calls_native'] for r in trials if r['forward_calls_native'] is not None]
        loop=[r['loop_token_passes'] for r in trials if r['loop_token_passes'] is not None]
        surfaces.append({
         'cohort':COHORTS[key][1],'cohort_key':key,'model':COHORTS[key][0],
         'action':action,'repair_trials':len(trials),'successful_trials':sum(r['task_success_reported'] for r in trials),
         'trial_success_fraction':sum(r['task_success_reported'] for r in trials)/len(trials),
         'distinct_failed_problems':n,'problems_with_any_success':succ,'problem_any_success_fraction':succ/n,
         'problem_wilson95_low':low,'problem_wilson95_high':hi,
         'mean_token_preservation':statistics.mean(pres) if pres else None,
         'mean_native_forward_calls':statistics.mean(comp) if comp else None,
         'mean_loop_token_passes':statistics.mean(loop) if loop else None
        })
write_csv('table03_cohort_action_surfaces.csv',surfaces)
for s in surfaces:
    if s['action'] in ('global','open_last_75','tail75'):
        print(f"{s['cohort']:<29s} {s['action']:<13s} trial={s['successful_trials']:>2}/{s['repair_trials']:<2} problem-any={s['problems_with_any_success']:>2}/{s['distinct_failed_problems']}")

## 4. Paired problem-level comparisons

For each cohort, compare each selective action to global regeneration on the **same original problems**. Within each problem/action, calculate the observed success fraction over all repair seeds. Compare paired fractions by exact sign tests, with ties excluded. Apply Holm correction within each cohort. No cross-model significance tests are asserted where datasets and intervention semantics differ.

In [ ]:
def exact_sign_p(plus,minus):
    n=plus+minus
    if n==0:return 1.0
    return min(1.,2*sum(math.comb(n,k) for k in range(0,min(plus,minus)+1))/2**n)

def holm_fix(rows):
    indexed=sorted(enumerate(rows),key=lambda a:a[1]['exact_two_sided_sign_p'])
    running=0
    for rank,(idx,row) in enumerate(indexed):
        running=max(running,min(1.,(len(rows)-rank)*row['exact_two_sided_sign_p']))
        rows[idx]['within_cohort_holm_p']=running

paired_tests=[]
for key in REPAIR_KEYS:
    rows=[r for r in ledger if r['cohort_key']==key and (key!='dream13' or r['role']=='primary_unseen_problem')]
    surf=problem_action_rows(rows)
    actions=sorted({r['action'] for r in rows if r['action']!='global'})
    coh=[]
    for action in actions:
        probs=sorted({p for p,a in surf if a==action} & {p for p,a in surf if a=='global'})
        diffs=[surf[p,action]['trial_success']-surf[p,'global']['trial_success'] for p in probs]
        plus=sum(x>1e-12 for x in diffs);minus=sum(x<-1e-12 for x in diffs)
        pval=exact_sign_p(plus,minus)
        # Bootstrap the effect over independent *problem IDs*.
        rng=random.Random(24500+len(coh)+len(paired_tests)*3)
        bs=[]
        for _ in range(2000):
            bs.append(statistics.mean(rng.choice(diffs) for _ in diffs))
        bs.sort()
        coh.append({
          'cohort':COHORTS[key][1],'selective_action':action,'n_paired_problems':len(probs),
          'paired_mean_success_difference':statistics.mean(diffs),
          'positive_problem_differences':plus,'negative_problem_differences':minus,
          'tied_problems':len(probs)-plus-minus,
          'exact_two_sided_sign_p':pval,
          'problem_bootstrap95_low':bs[49],
          'problem_bootstrap95_high':bs[1949],
        })
    holm_fix(coh)
    paired_tests.extend(coh)
write_csv('table04_problem_level_paired_tests.csv',paired_tests)
print('Paired problem-level tests:',len(paired_tests))
for r in paired_tests:
    if r['cohort']=='Dream independent':print(r)

## 5. Cross-model evidence matrix

Show the descriptive best fixed intervention versus global per source cohort, while distinguishing in-sample outcomes and model-native intervention semantics. In-sample oracle values are intentionally not used as evidence of a learnable selector.

In [ ]:
matrix=[]
for key in REPAIR_KEYS:
    ss=[s for s in surfaces if s['cohort_key']==key]
    fixed=sorted(ss,key=lambda r:(-r['trial_success_fraction'],r['action']))[0]
    global_row=next(r for r in ss if r['action']=='global')
    # In-sample oracle: a problem is counted if any action/seed succeeds.
    source=[r for r in ledger if r['cohort_key']==key and (key!='dream13' or r['role']=='primary_unseen_problem')]
    probs=defaultdict(list)
    for r in source:probs[r['problem_id']].append(r['task_success_reported'])
    oracle=sum(any(v) for v in probs.values())/len(probs)
    matrix.append({
       'cohort':COHORTS[key][1], 'model':COHORTS[key][0],
       'precision':COHORTS[key][2], 'native_intervention':COHORTS[key][3],
       'decoder':COHORTS[key][4],
       'n_distinct_failure_problems':len(probs),
       'n_repair_trials':len(source),'best_fixed_action':fixed['action'],
       'best_fixed_trial_rate':fixed['trial_success_fraction'],
       'global_trial_rate':global_row['trial_success_fraction'],
       'best_fixed_minus_global':fixed['trial_success_fraction']-global_row['trial_success_fraction'],
       'in_sample_problem_oracle_upper_bound':oracle,
       'data_role':'confirmatory_primary' if key=='dream13' else 'exploratory',
    })
write_csv('table05_evidence_matrix.csv',matrix)
for r in matrix:print(r['cohort'], 'best=',r['best_fixed_action'],round(100*r['best_fixed_trial_rate'],2),'global=',round(100*r['global_trial_rate'],2))

## 6. Manual verification queue

Preserve the 140 already flagged records and add novel BD3LM/MDLM records with verifier disagreements as a separate supplementary queue. Automated checks do **not** adjudicate explanatory correctness or whether the answer was intended as the final response.

In [ ]:
write_csv('manual_review_140_original.csv',review_queue)
new_flags=[]
for src,recheck in zip(ledger,sensitivity):
    if src['cohort_key'] not in ('bd3lm23','alodlm16_left1','mdlm18','bd3lm19'):continue
    if not recheck['requires_human_adjudication'] and not (src['task_success_reported'] and not recheck['explicit_final_correct']):continue
    new_flags.append({
       'cohort':src['cohort'],'problem_id':src['problem_id'],'action':src['action'],
       'trial_id':src['trial_id'],'dataset':src['dataset'],
       'reported_task_success':src['task_success_reported'],
       'exact_expression_witness':recheck['exact_expression_witness'],
       'explicit_final_correct':recheck['explicit_final_correct'],
       'source_output':src['original_output'],
       'repaired_output':src['repaired_output'],
       'human_decision':'UNREVIEWED','reviewer':'','review_note':''
    })
write_csv('manual_review_supplement_new_cases.csv',new_flags)
print('Original review cases:',len(review_queue),'| additional flagged cases:',len(new_flags))
print('Both files are unadjudicated templates.')

## 7. Optional MDLM Notebook 21 repair results (later)

The **generation-only** file already exists. The MDLM controlled-denoising repair ZIP is not yet available in the evidence provided. Do not interpret missing data as 0/64. To extend the notebook later, place the final results ZIP in Colab and set `OPTIONAL_MDLM21_REPAIR_ZIP` to its path. This will *append a separate cohort*; it never alters the frozen 710 source trials.

In [ ]:
OPTIONAL_MDLM21_REPAIR_ZIP=''  # e.g. '/content/RepairScopeX_NB21_MDLM_ConstantPerBlock_Repair_results.zip'
optional_status={'state':'pending','source':'MDLM NB21 controlled-denoising repair'}
if OPTIONAL_MDLM21_REPAIR_ZIP:
    p=Path(OPTIONAL_MDLM21_REPAIR_ZIP)
    assert p.is_file(),f'Optional source not found: {p}'
    with zipfile.ZipFile(p) as z:
        names=z.namelist()
        assert 'repair_trials.jsonl' in names, f'Missing repair_trials.jsonl: {names}'
        data=[json.loads(s) for s in z.read('repair_trials.jsonl').decode().splitlines() if s.strip()]
    ids=[r.get('id') or r.get('trial_id') for r in data]
    assert len(data)==64 and all(ids) and len(set(ids))==64
    assert all(r.get('status')=='ok' for r in data)
    optional_status={'state':'complete','n_repair_trials':64,'task_successes':sum(bool(r.get('task_success')) for r in data),
        'sha256':hashlib.sha256(p.read_bytes()).hexdigest()}
    write_csv('optional_mdlm21_raw_repair_results.csv',data)
print('MDLM Notebook 21 repair:',optional_status)

## 8. Publication figures

Figures describe different experiments without implying directly comparable compute units. All figure labels refer to the **reported** task-success metric, which must be complemented by the verification audit.

In [ ]:
try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
except Exception as exc:
    plt=None
    print('Matplotlib unavailable; CSV tables are still generated:',exc)

if plt:
    fig,ax=plt.subplots(figsize=(10,5.4))
    names=[x['cohort'] for x in matrix]
    y=[100*x['best_fixed_trial_rate'] for x in matrix]
    g=[100*x['global_trial_rate'] for x in matrix]
    xx=list(range(len(names)))
    ax.bar([i-.19 for i in xx],y,width=.38,label='Best fixed (in-sample)')
    ax.bar([i+.19 for i in xx],g,width=.38,label='Global')
    ax.set_xticks(xx,names,rotation=27,ha='right')
    ax.set_ylabel('Reported trial-level task success (%)')
    ax.set_title('RepairScope-X: Cohort-specific repair outcomes')
    ax.legend(frameon=False)
    ax.set_ylim(0,max(y+g+[1])*1.3)
    fig.tight_layout();fig.savefig(OUT/'fig01_repair_cohort_success.png',dpi=180);fig.savefig(OUT/'fig01_repair_cohort_success.pdf');plt.close(fig)

    fig,ax=plt.subplots(figsize=(9,5.4))
    models=sorted({r['model'] for r in surfaces})
    for model in models:
        rr=[r for r in surfaces if r['model']==model and r['mean_token_preservation'] is not None]
        ax.scatter([100*r['mean_token_preservation'] for r in rr],
                   [100*r['trial_success_fraction'] for r in rr],label=model,s=46)
    ax.set_xlabel('Mean token-position preservation (%)')
    ax.set_ylabel('Reported repair success (%)')
    ax.set_title('Preservation versus success (exploratory; not causal)')
    ax.legend(frameon=False,fontsize=9);fig.tight_layout()
    fig.savefig(OUT/'fig02_preservation_success.png',dpi=180)
    fig.savefig(OUT/'fig02_preservation_success.pdf');plt.close(fig)

    fig,ax=plt.subplots(figsize=(8.5,4.6))
    compare=[s for s in surfaces if s['cohort_key']=='bd3lm23']
    xx=list(range(len(compare)))
    ax.bar(xx,[x['mean_native_forward_calls'] for x in compare])
    ax.set_xticks(xx,[r['action'] for r in compare])
    ax.set_ylabel('Mean native forward calls')
    ax.set_title('BD3LM controlled-denoising: compute by repair scope')
    fig.tight_layout();fig.savefig(OUT/'fig03_bd3lm_native_compute.png',dpi=180)
    fig.savefig(OUT/'fig03_bd3lm_native_compute.pdf');plt.close(fig)
    print('Figures saved (3 PNG + 3 PDF).')

## 9. Integrity gate and final ZIP download

Gate checks only reproducible bookkeeping and presence of source records; a passing audit **is not evidence of causal superiority** or human-validated ground truth.

In [ ]:
report={
 'generated_utc':datetime.now(timezone.utc).isoformat(),
 'frozen_bundle_sha256':BUNDLE_SHA256,
 'repair_trials':len(ledger),
 'cohorts':len(REPAIR_KEYS),
 'primary_dream_confirmatory_trials':sum(r['cohort_key']=='dream13' and r['role']=='primary_unseen_problem' for r in ledger),
 'original_manual_review_cases':len(review_queue),
 'additional_manual_review_flags':len(new_flags),
 'optional_mdlm21_repair':optional_status,
 'inference_unit':'distinct problem_id within each cohort',
 'source_provenance':PROVENANCE,
 'all_records_status_ok':all(r['status']=='ok' for r in ledger),
 'NO_DUPLICATES':len({(r['cohort_key'],r['trial_id']) for r in ledger})==len(ledger)
}
report['AUDIT_PASS']=(report['repair_trials']==710 and report['cohorts']==7 and
                      report['primary_dream_confirmatory_trials']==120 and
                      report['original_manual_review_cases']==140 and
                      report['all_records_status_ok'] and report['NO_DUPLICATES'])
write_json('NB24_GATE.json',report)
assert report['AUDIT_PASS'],report
print('NOTEBOOK 24 INTEGRITY GATE: PASS')
print(json.dumps({k:v for k,v in report.items() if k not in ['source_provenance']},indent=2))

In [ ]:
lines=[
 '# RepairScope-X — Reconstructed Evidence Interpretation',
 '',
 '## Scope',
 f"- Verified completed repair records: {len(ledger)} across {len(REPAIR_KEYS)} separate cohorts.",
 '- Dream Notebook 13 primary confirmation: 120 trials; its 30 secondary trials stay out of primary comparisons.',
 '- ALoDLM entropy and left1 are distinct decoder conditions. The entropy reference copy is excluded.',
 '- New BD3LM constant-per-block results are included. The MDLM Notebook 21 repair results are pending.',
 '- No pooled cross-model success proportion is calculated.',
 '',
 '## Cohort-specific evidence',
]
for x in matrix:
 lines.append(f"- {x['cohort']}: {x['n_repair_trials']} trials on {x['n_distinct_failure_problems']} problems; "
 f"best fixed {x['best_fixed_action']} {x['best_fixed_trial_rate']:.1%}; global {x['global_trial_rate']:.1%}.")
lines+=['', '## Limitations','- Some task-success labels rely on heuristic extraction of the final number.','- Countdown expression witnesses must not be confused with complete final answers.',
 '- Repeated seeds are nested under a problem. Problem-level uncertainty and paired tests are primary.',
 '- Dream and ALoDLM have different repair mechanisms and different native compute proxies.',
 '- The tested model sizes, precisions, benchmarks and frozen failure cohorts are not identical.',
 '- In-sample oracle values overestimate realizable automatic selection.',
 f'- The original 140 manual-review cases and {len(new_flags)} additional flags remain unadjudicated.',
 '', '## Next decision',
 'Obtain Notebook 21 MDLM controlled-denoising repair results; review the flagged Countdown and final-answer outputs; then freeze tables and manuscript framing.',
]
(OUT/'RESEARCH_INTERPRETATION.md').write_text('\n'.join(lines)+'\n',encoding='utf-8')

artifacts=sorted(p for p in OUT.iterdir() if p.is_file() and p.name!='SHA256_MANIFEST.json')
write_json('SHA256_MANIFEST.json',{p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in artifacts})
archive=ROOT/'RepairScopeX_NB24_CrossModel_Evidence_results.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED,compresslevel=7) as z:
    for p in sorted(OUT.iterdir()):
        if p.is_file():z.write(p,arcname=p.name)
with zipfile.ZipFile(archive) as z:
    manifest=json.loads(z.read('SHA256_MANIFEST.json'))
    assert all(hashlib.sha256(z.read(k)).hexdigest()==v for k,v in manifest.items())
print('Created:',archive, '| size:',round(archive.stat().st_size/1024,1),'KiB')
try:
    from google.colab import files
    files.download(str(archive))
except ImportError:
    print('Outside Colab: ZIP available at',archive)

### Scientific caution

Notebook 24 is **synthesis and adjudication preparation**, not a new GPU inference experiment. The existing 140 cases are still awaiting manual review. If the stricter audit changes individual reported labels, tables must be regenerated from corrected adjudications rather than silently presenting the old rates as verified.